In [4]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import gymnasium as gym

In [5]:
# Create the environment
env = make_vec_env('LunarLander-v2', n_envs=16)

In [6]:
# SOLUTION
# We added some parameters to accelerate the training
model = PPO(
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 10,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

Using cpu device


In [ ]:
# SOLUTION
# Train it for 1,000,000 timesteps
model.learn(total_timesteps=1000000)
# Save the model
model_name = "ppo-LunarLander-v2-2"
model.save(model_name)

In [14]:
#@title
eval_env = Monitor(gym.make("LunarLander-v2", render_mode='rgb_array'))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=283.43 +/- 18.39498910278891


In [ ]:
model = PPO.load("ppo-LunarLander-v2")
obs = env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, dones, info = env.step(action)
    env.render()

KeyboardInterrupt: 

# Creating Custom Environment for Different Games

To use PPO with your own game character, you need to create a custom Gymnasium environment where you define:
1. **Actions**: What the character can do
2. **States**: What information the character sees
3. **Rewards**: How to score the character's performance

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class CustomGameEnv(gym.Env):
    """Custom Environment for your game character"""
    
    def __init__(self):
        super(CustomGameEnv, self).__init__()
        
        # Define ACTION SPACE - what your character can do
        # Example: 4 actions [move_left, move_right, jump, attack]
        self.action_space = spaces.Discrete(4)
        
        # Define OBSERVATION SPACE - what your character sees
        # Example: [x_position, y_position, enemy_distance, health]
        self.observation_space = spaces.Box(
            low=np.array([0, 0, 0, 0]), 
            high=np.array([100, 100, 50, 100]), 
            dtype=np.float32
        )
        
        # Initialize game state
        self.character_x = 50
        self.character_y = 0
        self.enemy_distance = 30
        self.health = 100
        
    def step(self, action):
        """Execute one action and return new state, reward, done, info"""
        
        # ACTIONS - Define what each action does
        if action == 0:  # move_left
            self.character_x = max(0, self.character_x - 5)
        elif action == 1:  # move_right
            self.character_x = min(100, self.character_x + 5)
        elif action == 2:  # jump
            self.character_y = 10 if self.character_y == 0 else 0
        elif action == 3:  # attack
            if self.enemy_distance < 10:
                self.enemy_distance = 50  # Enemy defeated
                
        # Update game state
        self.enemy_distance = max(0, self.enemy_distance - 1)  # Enemy moves closer
        
        # REWARDS - Define how to score actions
        reward = 0
        if action == 3 and self.enemy_distance > 40:  # Successful attack
            reward = 10
        elif self.enemy_distance == 0:  # Enemy reached character
            self.health -= 10
            reward = -5
        else:
            reward = -0.1  # Small penalty for each step (encourages efficiency)
            
        # STATES - Current game state
        observation = np.array([
            self.character_x, 
            self.character_y, 
            self.enemy_distance, 
            self.health
        ], dtype=np.float32)
        
        # Episode termination conditions
        terminated = self.health <= 0  # Game over
        truncated = False  # Could add time limit here
        info = {}
        
        return observation, reward, terminated, truncated, info
    
    def reset(self, seed=None, options=None):
        """Reset the environment to initial state"""
        super().reset(seed=seed)
        
        self.character_x = 50
        self.character_y = 0
        self.enemy_distance = 30
        self.health = 100
        
        observation = np.array([
            self.character_x, 
            self.character_y, 
            self.enemy_distance, 
            self.health
        ], dtype=np.float32)
        
        return observation, {}
    
    def render(self):
        """Optional: Display the game state"""
        print(f"Character: ({self.character_x}, {self.character_y}), Enemy Distance: {self.enemy_distance}, Health: {self.health}")

In [ ]:
# Now you can use PPO with YOUR custom game!
custom_env = CustomGameEnv()

# Train PPO on your custom environment
custom_model = PPO(
    policy='MlpPolicy',
    env=custom_env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1
)

print("Training PPO on custom game environment...")
custom_model.learn(total_timesteps=10000)

# Test the trained model
obs, info = custom_env.reset()
for _ in range(10):
    action, _states = custom_model.predict(obs)
    obs, reward, terminated, truncated, info = custom_env.step(action)
    custom_env.render()
    print(f"Action: {action}, Reward: {reward}")
    
    if terminated or truncated:
        obs, info = custom_env.reset()
        print("Game reset!")

## Key Points for Custom Environments:

1. **Actions (`action_space`)**: Define what your character can do
   - `spaces.Discrete(n)` for discrete actions (like button presses)
   - `spaces.Box()` for continuous actions (like joystick movement)

2. **States (`observation_space`)**: Define what information your character sees
   - Position, health, enemy locations, items, etc.
   - Must match the array you return in `step()` and `reset()`

3. **Rewards**: Define in the `step()` method
   - Positive rewards for good actions
   - Negative rewards for bad actions
   - Shape rewards to guide learning

4. **Same PPO code works**: Just change `env=custom_env` instead of `env=lunar_lander_env`

The beauty is that **PPO doesn't care what game it is** - it just needs the environment to follow the Gymnasium interface!